### 패키지 설치

In [6]:
#%pip install pandas openpyxl

### 라이브러리 불러오기

In [12]:
from pathlib import Path
import pandas as pd

### 파일 위치와 시트 확인

In [13]:
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
file_path = project_root / "data" / "raw" / "online_retail_II.xlsx"

if not file_path.is_file():
    raise FileNotFoundError(f"원본 파일 위치를 확인하세요: {file_path}")

excel = pd.ExcelFile(file_path, engine="openpyxl")

print("파일: ", file_path)
print("시트 목록: ", excel.sheet_names)

파일:  /Users/seongjinseo/projects/ecommerce-customer-analysis/data/raw/online_retail_II.xlsx
시트 목록:  ['Year 2009-2010', 'Year 2010-2011']


##### 원본 구조 확인 (첫 5행)

In [9]:
for sheet in excel.sheet_names:
    sample = pd.read_excel(excel, sheet_name=sheet, nrows=5)

    print(f"\n시트: {sheet}")
    print(f"컬럼: ", sample.columns.tolist())

    display(sample)
    display(sample.dtypes.to_frame("샘플에서 추정한 자료형"))


시트: Year 2009-2010
컬럼:  ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


,샘플에서 추정한 자료형
Invoice,int64
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,int64
Country,object



시트: Year 2010-2011
컬럼:  ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


,샘플에서 추정한 자료형
Invoice,int64
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,int64
Country,object


### 데이터 확인

원본 파일의 시트별 전체 행 수, 날짜 범위를 확인한다.

In [10]:
id_types = {
    "Invoice": "string",
    "StockCode": "string",
    "Customer ID": "string",
}

frames = []
summary = []

for sheet in excel.sheet_names:
    df = pd.read_excel(
        excel,
        sheet_name=sheet,
        dtype=id_types, # 문자형으로 강제로 지정해서 가져오기 (취소건도 있음)
    )

    # 기존 값은 유지하면서 날짜로 해석 가능한지 확인
    dates = pd.to_datetime(df['InvoiceDate'], errors="coerce")

    summary.append({
        "sheet": sheet,
        "rows": len(df),
        "columns": len(df.columns),
        "start_date": dates.min(),
        "end_date": dates.max(),
        "missing_date_rows": df["InvoiceDate"].isna().sum(),
        "invalid_date_rows": (
            df["InvoiceDate"].notna() & dates.isna()
        ).sum(),
    })

    # 합친 뒤에도 어느 시트에서 왔는지 추적
    df["source_sheet"] = sheet
    frames.append(df)

sheet_summary = pd.DataFrame(summary)
raw_df = pd.concat(frames, ignore_index=True)

display(sheet_summary)

print(f"합친 전체 행 수: {len(raw_df):,}")
print(f"원본 컬럼 수: {len(raw_df.columns) - 1}")
print(f"추가한 추적 컬럼 수: source_sheet")

KeyboardInterrupt: 

In [ ]:
structure_summary = pd.DataFrame({
    "dtype": raw_df.dtypes.astype(str),
    "missing_rows": raw_df.isna().sum(),
    "missing_pct": raw_df.isna().mean().mul(100).round(2),
})

display(structure_summary)

,dtype,missing_rows,missing_pct
Invoice,string,0,0.00
StockCode,string,0,0.00
Description,object,4382,0.41
Quantity,int64,0,0.00
InvoiceDate,datetime64[ns],0,0.00
Price,float64,0,0.00
Customer ID,string,243007,22.77
Country,object,0,0.00
source_sheet,object,0,0.00


고객 번호 누락, 중복 행, 수량·단가의 0 및 음수를 점검한다.

In [ ]:
original_cols = [
    "Invoice", "StockCode", "Description", "Quantity", 
    "InvoiceDate", "Price", "Customer ID", "Country",
]

# 1. 각 시트 내부에서 첫 행을 제외한 완전 중복 행 수
within_sheet = (
    raw_df
    .groupby("source_sheet", sort=False)[original_cols]
    .apply(lambda x: x.duplicated().sum())
    .rename("duplicate_extra_rows")
)

display(within_sheet.to_frame())

# 2. 같은 8개 컬럼 조합이 서로 다른 시트에도 있는지 확인
# dropna=False: Customer ID나 Description이 누락된 조합도 비교에 포함
row_groups = (
    raw_df
    .groupby(original_cols, dropna=False)
    .agg(
        row_count=("source_sheet", "size"),
        sheet_count=("source_sheet", "nunique"),
    )
    .reset_index()
)

cross_sheet = row_groups.loc[row_groups["sheet_count"] > 1]

print(f"두 시트에 공통으로 존재하는 행 조합: {len(cross_sheet):,}")
print(f"해당 조합에 속한 전체 행 수: {cross_sheet['row_count'].sum():,}")

display(cross_sheet.head(10))

,duplicate_extra_rows
source_sheet,
Year 2009-2010,6865
Year 2010-2011,5268


두 시트에 공통으로 존재하는 행 조합: 22,202
해당 조합에 속한 전체 행 수: 45,046


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,row_count,sheet_count
486538,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom,2,2
486539,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom,2,2
486540,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2,2
486541,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2,2
486542,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,2,2
486543,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,2,2
486544,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,2,2
486545,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,2,2
486546,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom,2,2
486547,536367,21754,HOME BUILDING BLOCK WORD,3,2010-12-01 08:34:00,5.95,13047,United Kingdom,2,2


In [ ]:
is_cancel = (
    raw_df['Invoice']
    .str.strip()
    .str.upper()
    .str.startswith("C", na=False)
)

quality_summary = pd.Series({
    "전체 행": len(raw_df),
    "취소 번호 행": is_cancel.sum(),
    "수량 0 이하 행": raw_df["Quantity"].le(0).sum(),
    "단가 0 이하 행": raw_df["Price"].le(0).sum(),
    "고객 ID 누락 행": raw_df["Customer ID"].isna().sum(),
    "상품명 누락 행": raw_df["Description"].isna().sum(),
}, name="rows")

display(quality_summary.to_frame())

,rows
전체 행,1067371
취소 번호 행,19494
수량 0 이하 행,22950
단가 0 이하 행,6207
고객 ID 누락 행,243007
상품명 누락 행,4382


두 시트는 2010.12.01~2010.12.09 기간이 겹치기 때문에 2010년 12월 1일부터는 두 번째 시트를 사용하는 방식이 가능한지 확인한다.

In [ ]:
cutoff = pd.Timestamp("2010-12-01")

# 첫 번째 시트에서 두 번째 시트와 기간이 겹치는 부분
first_overlap = raw_df.loc[
    raw_df["source_sheet"].eq("Year 2009-2010")
    & raw_df["InvoiceDate"].ge(cutoff),
    original_cols,
]

# 두 번째 시트 전체
second_sheet = raw_df.loc[
    raw_df["source_sheet"].eq("Year 2010-2011"),
    original_cols,
]

# 동일한 8개 컬럼 조합별 등장 횟수 비교
first_counts = (
    first_overlap
    .groupby(original_cols, dropna=False)
    .size()
    .rename("first_count")
    .reset_index()
)

second_counts = (
    second_sheet
    .groupby(original_cols, dropna=False)
    .size()
    .rename("second_count")
    .reset_index()
)

comparison = first_counts.merge(
    second_counts,
    on=original_cols,
    how="left",
    validate="one_to_one",
)

comparison["second_count"] = (
    comparison["second_count"].fillna(0).astype("int64")
)

only_first = comparison.loc[comparison["second_count"].eq(0)]

fewer_in_second = comparison.loc[
    comparison["second_count"].gt(0)
    & comparison["second_count"].lt(comparison["first_count"])
]

print(f"첫 시트의 중첩 기간 행 수: {len(first_overlap):,}")
print(f"첫 시트에만 있는 조합 수: {len(only_first):,}")
print(f"첫 시트에만 있는 조합의 행 수: {only_first['first_count'].sum():,}")
print(f"두 번째 시트에서 반복 횟수가 더 적은 조합 수: {len(fewer_in_second):,}")

display(only_first.head(10))
display(fewer_in_second.head(10))

첫 시트의 중첩 기간 행 수: 22,523
첫 시트에만 있는 조합 수: 0
첫 시트에만 있는 조합의 행 수: 0
두 번째 시트에서 반복 횟수가 더 적은 조합 수: 0


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,first_count,second_count


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,first_count,second_count


첫 시트에 있는 2010.12.01~2010.12.09 행이 모두 두 번째 시트와 겹침을 확인
- 2010-12-01 이전: first sheet 사용
- 2010-12-01 이후: second sheet 사용

In [ ]:
cutoff = pd.Timestamp("2010-12-01")

keep_rows = (
    (
        raw_df["source_sheet"].eq("Year 2009-2010")
        & raw_df["InvoiceDate"].lt(cutoff)
    )
    |
    (
        raw_df["source_sheet"].eq("Year 2010-2011")
        & raw_df["InvoiceDate"].ge(cutoff)
    )
)

base_df = raw_df.loc[keep_rows].copy().reset_index(drop=True)

print(f"원본 행 수: {len(raw_df):,}")
print(f"시트 중첩으로 제외한 행 수: {len(raw_df) - len(base_df):,}")
print(f"정리 후 행 수: {len(base_df):,}")

assert len(raw_df) - len(base_df) == 22_523
assert len(base_df) == 1_044_848

원본 행 수: 1,067,371
시트 중첩으로 제외한 행 수: 22,523
정리 후 행 수: 1,044,848


판매 분석 조건을 적용한 뒤 남는 데이터

In [ ]:
is_cancel = (
    base_df["Invoice"]
    .str.strip()
    .str.upper()
    .str.startswith("C", na=False)
)

sales_mask = (
    ~is_cancel
    & base_df['Quantity'].gt(0)
    & base_df['Price'].gt(0)
)

sales_df = base_df.loc[sales_mask].copy()

customer_sales_df = sales_df.loc[
    sales_df["Customer ID"].notna()
].copy()

summary = pd.Series({
    "시트 중첩 정리 후 행 수": len(base_df),
    "판매 조건에서 제외한 행 수": (~sales_mask).sum(),
    "판매 조건 충족 행 수": len(sales_df),
    "판매 조건 충족 중 고객 ID 누락 행 수": sales_df["Customer ID"].isna().sum(),
    "고객 분석 가능 행 수": len(customer_sales_df),
    "고객 분석 가능 고유 고객 수": customer_sales_df["Customer ID"].nunique(),
}, name="count")

display(summary.to_frame())

,count
시트 중첩 정리 후 행 수,1044848
판매 조건에서 제외한 행 수,25194
판매 조건 충족 행 수,1019654
판매 조건 충족 중 고객 ID 누락 행 수,228609
고객 분석 가능 행 수,791045
고객 분석 가능 고유 고객 수,5878


특수 상품 코드를 확인한다.

In [ ]:
codes = sales_df["StockCode"].str.strip()

special_mask = ~codes.str.fullmatch(
    r"\d{5}[A-Za-z]?",
    na=False,
)

special_summary = (
    sales_df.loc[special_mask]
    .groupby("StockCode", dropna=False)
    .agg(
        rows=("Invoice", "size"),
        orders=("Invoice", "nunique"),
        description_examples=(
            "Description",
            lambda s: list(s.dropna().unique()[:5]),
        ),
    )
    .sort_values("rows", ascending=False)
)

display(special_summary)

,rows,orders,description_examples
StockCode,,,
POST,1851,1851,[POSTAGE]
DOT,1415,1415,[DOTCOM POSTAGE]
15056BL,893,857,[EDWARDIAN PARASOL BLACK]
M,856,784,[Manual]
C2,267,267,[CARRIAGE]
79323LP,170,170,[LIGHT PINK CHERRY LIGHTS]
15056bl,87,87,[EDWARDIAN PARASOL BLACK]
79323GR,83,83,[GREEN CHERRY LIGHTS]
ADJUST,36,36,"[Adjustment by john on 26/01/2010 16, Adjustme..."


코드 분류표
- 배송료: `POST`, `DOT`, `C2`
- 수동 입력/조정: `M`, `m`, `ADJUST`, `ADJUST2`, `B`
- 수수료/할인: `BANK CHARGES`, `AMAZONFEE`, `D`
- 테스트/샘플: `TEST001`, `TEST002`, `S`
- 상품권: `gift_0001_*`
- 일반 상품: 그외

일반 상품만 유지, 나머지는 분석에서 제외<br>
이 금액은 취소 및 환불을 차감하지 않은 값이며, 시트 내부 동일 행도 유지한 상태

In [ ]:
classified_sales_df = sales_df.copy()

code_key = (
    classified_sales_df['StockCode']
    .str.strip()
    .str.upper()
)

code_groups = {
    "POST": "shipping",
    "DOT": "shipping",
    "C2": "shipping",
    "M": "manual_or_adjustment",
    "m": "manual_or_adjustment",
    "ADJUST": "manual_or_adjustment",
    "ADJUST2": "manual_or_adjustment",
    "B": "manual_or_adjustment",
    "BANK CHARGES": "fee_or_discount",
    "AMAZONFEE": "fee_or_discount",
    "D": "fee_or_discount",
    "TEST001": "test_or_sample",
    "TEST002": "test_or_sample",
    "S": "test_or_sample",
}

classified_sales_df["item_type"] = (
    code_key.map(code_groups).fillna("product")
)

classified_sales_df.loc[
    code_key.str.startswith("GIFT_0001_", na=False),
    "item_type",
] = "gift_voucher"

classified_sales_df["line_amount"] = (
    classified_sales_df["Quantity"]*classified_sales_df["Price"]
)

type_summary = (
    classified_sales_df
    .groupby("item_type")
    .agg(
        rows=("Invoice", "size"),
        orders=("Invoice", "nunique"),
        amount_gbp=("line_amount", "sum"),
    )
    .sort_values("amount_gbp", ascending=False)
)

display(type_summary.round({"amount_gbp": 2}))

,rows,orders,amount_gbp
item_type,,,
product,1015088,39516,19700954.46
shipping,3533,3533,448962.53
manual_or_adjustment,901,828,360320.98
fee_or_discount,42,41,21384.93
gift_voucher,77,70,1756.17
test_or_sample,13,13,362.85


In [ ]:
product_sales_df = classified_sales_df.loc[
    classified_sales_df["item_type"].eq("product")
].copy()

customer_product_sales_df = product_sales_df.loc[
    product_sales_df["Customer ID"].notna()
].copy()

### SQLite에 적재

##### SQL에 맞게 컬럼명 정리

In [ ]:
db_df = base_df.rename(columns={
    "Invoice": "invoice",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country"
}).copy()

# 각 행을 구분하는 내부 번호 (invoice와 다름)
db_df.insert(0, "record_id", range(1, len(db_df) + 1))

# SQLite에는 날짜를 정렬 가능한 문자열로 지정
db_df["invoice_date"] = (
    db_df["invoice_date"].dt.strftime("%Y-%m-%d %H:%M:%S")
)

# code_groups 분류 기준을 전체 데이터에 적용
code_key = db_df["stock_code"].str.strip().str.upper()

db_df["item_type"] = code_key.map(code_groups).fillna("product")

db_df.loc[
    code_key.str.startswith("GIFT_0001_", na=False),
    "item_type",
] = "gift_voucher"

##### DB 생성 및 적재

In [14]:
import sqlite3

db_path = project_root / "data" / "ecommerce.db"
conn = sqlite3.connect(db_path)

In [ ]:
db_df.to_sql(
    "transactions",
    conn,
    if_exists="fail",
    index=False,
    chunksize=10_000,
    dtype={
        "record_id": "INTEGER",
        "invoice": "TEXT",
        "stock_code": "TEXT",
        "description": "TEXT",
        "quantity": "INTEGER",
        "invoice_date": "TEXT",
        "unit_price": "REAL",
        "customer_id": "TEXT",
        "country": "TEXT",
        "source_sheet": "TEXT",
        "item_type": "TEXT", 
    },
)

print("저장 위치: ", db_path)

##### 분석용 SQL뷰 생성

In [15]:
sql_path = project_root / "sql" / "01_create_views.sql"

conn.executescript(sql_path.read_text(encoding="utf-8"))
conn.commit()

##### 적재 결과 검증

In [16]:
verification = pd.read_sql_query("""
SELECT
    (SELECT COUNT(*) FROM transactions) AS base_rows,
    COUNT(*) AS product_rows,
    COUNT(DISTINCT invoice) AS product_orders,
    ROUND(SUM(line_amount), 2) AS product_amount_gbp
FROM product_sales; 
""", conn)

display(verification)

,base_rows,product_rows,product_orders,product_amount_gbp
0,1044848,1015088,39516,19700954.46


월별 상품 판매액, 주문 수, 객단가 확인

In [17]:
sql_path = project_root / "sql" / "02_monthly_sales.sql"

monthly_sales = pd.read_sql_query(
    sql_path.read_text(encoding="utf-8"),
    conn,
)

display(monthly_sales)

,month,orders,sales_gbp,avg_order_value_gbp
0,2009-12,1666,801102.97,480.85
1,2010-01,1049,613850.81,585.18
2,2010-02,1189,539744.69,453.95
3,2010-03,1647,764333.85,464.08
4,2010-04,1435,648993.82,452.26
5,2010-05,1484,645645.39,435.07
6,2010-06,1612,699316.02,433.82
7,2010-07,1507,634896.99,421.30
8,2010-08,1402,676060.72,482.21
9,2010-09,1789,871878.34,487.36


월별 지표를 전년 같은 월과 비교

In [18]:
sql_path = project_root / "sql"/ "03_monthly_yoy.sql"

monthly_yoy = pd.read_sql_query(
    sql_path.read_text(encoding="'utf-8"),
    conn,
)

display(monthly_yoy)

,month,orders,sales_gbp,avg_order_value_gbp,previous_year_sales_gbp,sales_yoy_pct,orders_yoy_pct,aov_yoy_pct
0,2010-12,1550,777931.77,501.89,801102.97,-2.89,-6.96,4.37
1,2011-01,1081,671933.19,621.58,613850.81,9.46,3.05,6.22
2,2011-02,1093,508886.20,465.59,539744.69,-5.72,-8.07,2.56
3,2011-03,1440,691257.61,480.04,764333.85,-9.56,-12.57,3.44
4,2011-04,1235,516218.12,417.99,648993.82,-20.46,-13.94,-7.58
5,2011-05,1668,741192.41,444.36,645645.39,14.80,12.40,2.14
6,2011-06,1525,738751.87,484.43,699316.02,5.64,-5.40,11.67
7,2011-07,1452,689364.41,474.77,634896.99,8.58,-3.65,12.69
8,2011-08,1339,725513.50,541.83,676060.72,7.31,-4.49,12.36
9,2011-09,1818,1030475.36,566.82,871878.34,18.19,1.62,16.30


1. 판매액 증가율이 가장 높은 월은 2011년 9월이다. 판매액은 전년 동월 대비 18.19% 증가했고, 주문 수 증가율 1.62%보다 객단가 증가율 16.30%가 컸다.
2. 판매액 감소율이 가장 큰 월은 2011년 4월이다. 주문 수는 13.94% 감소했으며, 주문 수와 객단가가 함께 감소해 판매액이 20.46% 감소했다.
3. 2011년 6~8월은 객단가 상승이 주문 수 감소를 상쇄하면서 판매액이 증가했다.
4. 2011년 10월은 판매액이 거의 유지되었다. 주문 수 10.65% 감소를 객단가 12.80% 상승이 상쇄해 판매액이 0.79% 증가했다.